## 01 IMPORTS

In [2]:
from pathlib import Path
import sqlite3
import re
import unicodedata

import numpy as np
import pandas as pd

from IPython.display import display

## 02 PROJECT PATHS

In [ ]:
PROJECT_ROOT = Path(
    r"C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor"
)

DATA_DIR = PROJECT_ROOT / "Data"

CLEAN_DIR = DATA_DIR / "Clean"
TARGET_DIR = DATA_DIR / "Prediction_Targets"
FEATURE_DIR = DATA_DIR / "Features"
DATABASE_DIR = DATA_DIR / "Database"

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# INPUT FILES

HISTORICAL_DATA_PATH = (
    CLEAN_DIR
    / "cleaned_movies.csv"
)

TARGET_DATA_PATH = (
    TARGET_DIR
    / "prediction_targets_complete.csv"
)


# OUTPUT FILES

FEATURE_DB_PATH = (
    DATABASE_DIR
    / "movie_features.db"
)

HISTORICAL_FEATURE_PATH = (
    FEATURE_DIR
    / "historical_model_features.csv"
)

TARGET_FEATURE_PATH = (
    FEATURE_DIR
    / "prediction_target_features.csv"
)


print("Historical data:")
print(HISTORICAL_DATA_PATH)

print("\nPrediction targets:")
print(TARGET_DATA_PATH)

print("\nSQLite feature database:")
print(FEATURE_DB_PATH)

Historical data:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Clean\cleaned_movies.csv

Prediction targets:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Prediction_Targets\prediction_targets_complete.csv

SQLite feature database:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Database\movie_features.db


## 03 LOADING HISTORICAL AND PREDICTION DATA

In [4]:
historical_movies = pd.read_csv(
    HISTORICAL_DATA_PATH
)

prediction_targets = pd.read_csv(
    TARGET_DATA_PATH,
    parse_dates=[
        "release_date"
    ]
)


print("=" * 80)
print("FEATURE ENGINEERING INPUTS")
print("=" * 80)

print(
    f"Historical movies: "
    f"{historical_movies.shape}"
)

print(
    f"Prediction targets: "
    f"{prediction_targets.shape}"
)


display(
    historical_movies.head()
)

display(
    prediction_targets[
        [
            "title",
            "cohort",
            "release_date",
            "budget",
            "genre",
            "director"
        ]
    ].head()
)

FEATURE ENGINEERING INPUTS
Historical movies: (7668, 20)
Prediction targets: (19, 61)


,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime,Profit,ROI,released_month,release_quarter,decade
0,The Shining,R,Drama,1980,1980-06-13,8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0,27998772.0,1.473620,1980-06-13,2.0,1980
1,The Blue Lagoon,R,Adventure,1980,1980-07-02,5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0,54353106.0,12.078468,1980-07-02,3.0,1980
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,1980-06-20,8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0,520375067.0,28.909726,1980-06-20,2.0,1980
3,Airplane!,PG,Comedy,1980,1980-07-02,7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0,79953539.0,22.843868,1980-07-02,3.0,1980
4,Caddyshack,R,Comedy,1980,1980-07-25,7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0,33846344.0,5.641057,1980-07-25,3.0,1980


,title,cohort,release_date,budget,genre,director
0,Spider-Man: Brand New Day,released_backtest,2026-07-31,225000000.0,"Action, Adventure, Fantasy",Destin Daniel Cretton
1,Michael,released_backtest,2026-04-24,170000000.0,"Biography, Drama, Music",Antoine Fuqua
2,The Odyssey,released_backtest,2026-07-17,250000000.0,"Adventure, Action, Fantasy",Christopher Nolan
3,Scary Movie,released_backtest,2026-06-05,30000000.0,"Comedy, Horror",Michael Tiddes
4,Mortal Kombat II,released_backtest,2026-05-08,80000000.0,"Action, Adventure, Fantasy",Simon McQuoid


## 04 FEATURE POLICY

In [5]:
feature_policy = pd.DataFrame({

    "feature": [
        "budget",
        "genre",
        "production_company",
        "director",
        "lead_star",
        "release_month",
        "release_quarter",
        "release_year",
        "holiday_release",
        "director_track_record",
        "director_prior_films",
        "star_track_record",
        "star_prior_films",
        "is_franchise",
        "is_sequel",
        "franchise_gap_years",
        "pre_release_sentiment",
        "social_interest"
    ],

    "use_status": [
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "REVIEW",
        "REVIEW",
        "REVIEW",
        "LATER",
        "LATER"
    ],

    "reason": [
        "Known or estimable before release.",
        "Known before release.",
        "Known before release.",
        "Known before release.",
        "Known before release.",
        "Derived from announced release date.",
        "Derived from announced release date.",
        "Derived from announced release date.",
        "Derived from announced release date.",
        "Can be calculated using prior films only.",
        "Can be calculated using prior films only.",
        "Can be calculated using prior films only.",
        "Can be calculated using prior films only.",
        "Needs comparable historical franchise data.",
        "Needs comparable historical sequel data.",
        "Needs comparable historical franchise-gap data.",
        "Historical equivalent not yet available.",
        "Historical equivalent not yet available."
    ]
})


print("=" * 80)
print("FEATURE POLICY")
print("=" * 80)

display(feature_policy)

FEATURE POLICY


,feature,use_status,reason
0,budget,USE,Known or estimable before release.
1,genre,USE,Known before release.
2,production_company,USE,Known before release.
3,director,USE,Known before release.
4,lead_star,USE,Known before release.
5,release_month,USE,Derived from announced release date.
6,release_quarter,USE,Derived from announced release date.
7,release_year,USE,Derived from announced release date.
8,holiday_release,USE,Derived from announced release date.
9,director_track_record,USE,Can be calculated using prior films only.


## 05 BUILD CORE HISTORICAL FEATURE TABLE

In [ ]:
historical_features = pd.DataFrame()


# IDENTITY

historical_features["title"] = (
    historical_movies["name"]
)

historical_features["release_year"] = (
    pd.to_numeric(
        historical_movies["year"],
        errors="coerce"
    )
    .astype("Int64")
)


# RELEASE DATE

historical_features["release_date"] = (
    pd.to_datetime(
        historical_movies["released"],
        errors="coerce"
    )
)


# FINANCIALS

historical_features["budget"] = (
    pd.to_numeric(
        historical_movies["budget"],
        errors="coerce"
    )
)

historical_features["worldwide_box_office"] = (
    pd.to_numeric(
        historical_movies["gross"],
        errors="coerce"
    )
)


# CORE CATEGORICAL FEATURES

historical_features["genre"] = (
    historical_movies["genre"]
)

historical_features["production_company"] = (
    historical_movies["company"]
)

historical_features["director"] = (
    historical_movies["director"]
)

historical_features["lead_star"] = (
    historical_movies["star"]
)


# OPTIONAL PRE-RELEASE CHARACTERISTICS

historical_features["runtime"] = (
    pd.to_numeric(
        historical_movies["runtime"],
        errors="coerce"
    )
)

historical_features["content_rating"] = (
    historical_movies["rating"]
)


print("=" * 80)
print("CORE HISTORICAL FEATURE TABLE")
print("=" * 80)

print(
    f"Rows: "
    f"{len(historical_features):,}"
)

print(
    f"Columns: "
    f"{len(historical_features.columns)}"
)


display(
    historical_features.head()
)

CORE HISTORICAL FEATURE TABLE
Rows: 7,668
Columns: 11


,title,release_year,release_date,budget,worldwide_box_office,genre,production_company,director,lead_star,runtime,content_rating
0,The Shining,1980,1980-06-13,19000000.0,46998772.0,Drama,Warner Bros.,Stanley Kubrick,Jack Nicholson,146.0,R
1,The Blue Lagoon,1980,1980-07-02,4500000.0,58853106.0,Adventure,Columbia Pictures,Randal Kleiser,Brooke Shields,104.0,R
2,Star Wars: Episode V - The Empire Strikes Back,1980,1980-06-20,18000000.0,538375067.0,Action,Lucasfilm,Irvin Kershner,Mark Hamill,124.0,PG
3,Airplane!,1980,1980-07-02,3500000.0,83453539.0,Comedy,Paramount Pictures,Jim Abrahams,Robert Hays,88.0,PG
4,Caddyshack,1980,1980-07-25,6000000.0,39846344.0,Comedy,Orion Pictures,Harold Ramis,Chevy Chase,98.0,R


## 06 BUILD CORE PREDICTION FEATURE TABLE

In [7]:
prediction_features = pd.DataFrame()


# IDENTITY

prediction_features["title"] = (
    prediction_targets["title"]
)

prediction_features["cohort"] = (
    prediction_targets["cohort"]
)

prediction_features["release_date"] = (
    pd.to_datetime(
        prediction_targets["release_date"],
        errors="coerce"
    )
)

prediction_features["release_year"] = (
    prediction_features[
        "release_date"
    ].dt.year.astype("Int64")
)


# FINANCIAL INPUT

prediction_features["budget"] = (
    pd.to_numeric(
        prediction_targets["budget"],
        errors="coerce"
    )
)


# CATEGORICAL INPUTS

prediction_features["genre"] = (
    prediction_targets["genre"]
)

prediction_features["production_company"] = (
    prediction_targets[
        "production_company"
    ]
)

prediction_features["director"] = (
    prediction_targets["director"]
)


# FIRST-BILLED LEAD STAR

prediction_features["lead_star"] = (
    prediction_targets[
        "lead_cast"
    ]
    .astype("string")
    .str.split(";")
    .str[0]
    .str.strip()
)


# KNOWN OUTCOME
# Only released/backtest movies have this.

prediction_features[
    "worldwide_box_office"
] = pd.to_numeric(
    prediction_targets[
        "worldwide_box_office"
    ],
    errors="coerce"
)


print("=" * 80)
print("CORE PREDICTION FEATURE TABLE")
print("=" * 80)

print(
    f"Rows: "
    f"{len(prediction_features):,}"
)

print(
    f"Columns: "
    f"{len(prediction_features.columns)}"
)


display(
    prediction_features[
        [
            "title",
            "cohort",
            "release_year",
            "budget",
            "genre",
            "production_company",
            "director",
            "lead_star",
            "worldwide_box_office"
        ]
    ]
)

CORE PREDICTION FEATURE TABLE
Rows: 19
Columns: 10


,title,cohort,release_year,budget,genre,production_company,director,lead_star,worldwide_box_office
0,Spider-Man: Brand New Day,released_backtest,2026,225000000.0,"Action, Adventure, Fantasy",Columbia Pictures; Pascal Pictures; Marvel Stu...,Destin Daniel Cretton,Tom Holland,2.402937e+09
1,Michael,released_backtest,2026,170000000.0,"Biography, Drama, Music",GK Films,Antoine Fuqua,Jaafar Jackson,1.021427e+09
2,The Odyssey,released_backtest,2026,250000000.0,"Adventure, Action, Fantasy",Syncopy; Universal Pictures,Christopher Nolan,Matt Damon,1.627387e+09
3,Scary Movie,released_backtest,2026,30000000.0,"Comedy, Horror",Miramax; Wayans Bros. Entertainment,Michael Tiddes,Anna Faris,2.312778e+08
4,Mortal Kombat II,released_backtest,2026,80000000.0,"Action, Adventure, Fantasy",New Line Cinema; Atomic Monster; Broken Road P...,Simon McQuoid,Karl Urban,1.294701e+08
5,The Super Mario Galaxy Movie,released_backtest,2026,110000000.0,"Animation, Adventure, Comedy, Action",Illumination Entertainment; Nintendo; Universa...,Aaron Horvath; Michael Jelenic; Pierre Leduc,Chris Pratt,1.012793e+09
6,The Devil Wears Prada 2,released_backtest,2026,100000000.0,"Comedy, Drama",Wendy Finerman Productions,David Frankel,Meryl Streep,6.928175e+08
7,Toy Story 5,released_backtest,2026,175000000.0,"Animation, Adventure, Comedy, Kids & Family",Pixar Animation Studios,Andrew Stanton,Tom Hanks,1.137320e+09
8,GOAT,released_backtest,2026,80000000.0,"Animation, Comedy, Adventure, Sports",Sony Pictures Animation; Unanimous Media,Tyree Dillihay,Caleb McLaughlin,1.953569e+08
9,Backrooms,released_backtest,2026,10000000.0,"Horror, Sci-Fi, Fantasy, Mystery & Thriller",A24; Chernin Entertainment; 21 Laps Entertainm...,Kane Parsons,Chiwetel Ejiofor,3.931673e+08


## 07 RELEASE TIMING FEATURES

In [8]:
def add_release_features(df):

    result = df.copy()

    result["release_month"] = (
        result[
            "release_date"
        ].dt.month
    )

    result["release_quarter"] = (
        result[
            "release_date"
        ].dt.quarter
    )


    # NORTH AMERICAN THEATRICAL SEASON

    def release_season(month):

        if pd.isna(month):
            return pd.NA

        if month in [5, 6, 7]:
            return "summer"

        if month in [11, 12]:
            return "holiday"

        if month in [9, 10]:
            return "fall"

        if month in [3, 4]:
            return "spring"

        return "winter"


    result["release_season"] = (
        result[
            "release_month"
        ].apply(
            release_season
        )
    )


    # HOLIDAY / EVENT RELEASE WINDOW

    result["holiday_release"] = (
        result[
            "release_month"
        ].isin(
            [11, 12]
        )
        .astype(int)
    )


    return result


historical_features = add_release_features(
    historical_features
)

prediction_features = add_release_features(
    prediction_features
)


print("=" * 80)
print("RELEASE FEATURES CREATED")
print("=" * 80)


display(
    prediction_features[
        [
            "title",
            "release_date",
            "release_month",
            "release_quarter",
            "release_season",
            "holiday_release"
        ]
    ]
)

RELEASE FEATURES CREATED


,title,release_date,release_month,release_quarter,release_season,holiday_release
0,Spider-Man: Brand New Day,2026-07-31,7,3,summer,0
1,Michael,2026-04-24,4,2,spring,0
2,The Odyssey,2026-07-17,7,3,summer,0
3,Scary Movie,2026-06-05,6,2,summer,0
4,Mortal Kombat II,2026-05-08,5,2,summer,0
5,The Super Mario Galaxy Movie,2026-04-03,4,2,spring,0
6,The Devil Wears Prada 2,2026-05-01,5,2,summer,0
7,Toy Story 5,2026-06-19,6,2,summer,0
8,GOAT,2026-02-13,2,1,winter,0
9,Backrooms,2026-05-29,5,2,summer,0


## 08 INITIALISING SQLITE FEATURE DATABASE

In [ ]:
connection = sqlite3.connect(
    FEATURE_DB_PATH
)


historical_features.to_sql(
    "historical_movies",
    connection,
    if_exists="replace",
    index=False
)


prediction_features.to_sql(
    "prediction_targets",
    connection,
    if_exists="replace",
    index=False
)


print("=" * 80)
print("SQLITE FEATURE DATABASE INITIALIZED")
print("=" * 80)

print(
    f"Database: "
    f"{FEATURE_DB_PATH}"
)


# VERIFY TABLES

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)


display(tables)

SQLITE FEATURE DATABASE INITIALIZED
Database: C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Database\movie_features.db


,name
0,historical_movies
1,prediction_targets


## 09 PREPARING STABLE IDs AND SQL MATCHING KEYS

In [10]:
def normalize_entity(value):
    #Normalize people/company names for matching.

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(char)
    )

    value = re.sub(
        r"[^a-z0-9\s]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value if value else pd.NA


# STABLE IDS

historical_features = (
    historical_features
    .reset_index(drop=True)
)

prediction_features = (
    prediction_features
    .reset_index(drop=True)
)

historical_features["historical_id"] = (
    np.arange(
        1,
        len(historical_features) + 1
    )
)

prediction_features["target_id"] = (
    np.arange(
        1,
        len(prediction_features) + 1
    )
)


# NORMALIZED DIRECTOR / STAR KEYS

historical_features["director_key"] = (
    historical_features["director"]
    .apply(normalize_entity)
)

historical_features["star_key"] = (
    historical_features["lead_star"]
    .apply(normalize_entity)
)

prediction_features["director_key"] = (
    prediction_features["director"]
    .apply(normalize_entity)
)

prediction_features["star_key"] = (
    prediction_features["lead_star"]
    .apply(normalize_entity)
)


# ------------------------------------------------------------
# PRIMARY PRODUCTION COMPANY
#
# Prediction targets can contain multiple companies separated
# by semicolons. We use the first-listed company as the
# comparable primary company.
# ------------------------------------------------------------

historical_features[
    "primary_production_company"
] = historical_features[
    "production_company"
]

prediction_features[
    "primary_production_company"
] = (
    prediction_features[
        "production_company"
    ]
    .astype("string")
    .str.split(";")
    .str[0]
    .str.strip()
)


historical_features["company_key"] = (
    historical_features[
        "primary_production_company"
    ]
    .apply(normalize_entity)
)

prediction_features["company_key"] = (
    prediction_features[
        "primary_production_company"
    ]
    .apply(normalize_entity)
)


# REWRITE SQLITE TABLES WITH IDS + KEYS

historical_features.to_sql(
    "historical_movies",
    connection,
    if_exists="replace",
    index=False
)

prediction_features.to_sql(
    "prediction_targets",
    connection,
    if_exists="replace",
    index=False
)


print("=" * 80)
print("SQL FEATURE KEYS PREPARED")
print("=" * 80)

print(
    f"Historical rows: "
    f"{len(historical_features):,}"
)

print(
    f"Prediction rows: "
    f"{len(prediction_features):,}"
)

print(
    f"Historical director keys: "
    f"{historical_features['director_key'].notna().sum():,}"
)

print(
    f"Historical star keys: "
    f"{historical_features['star_key'].notna().sum():,}"
)

SQL FEATURE KEYS PREPARED
Historical rows: 7,668
Prediction rows: 19
Historical director keys: 7,668
Historical star keys: 7,667


## 10 DIRECTOR HISTORICAL PERFORMANCE WITH SQLITE

In [ ]:
cursor = connection.cursor()


# HISTORICAL MOVIE DIRECTOR FEATURES

cursor.execute(
    "DROP TABLE IF EXISTS director_features;"
)

cursor.execute(
    """
    CREATE TABLE director_features AS

    SELECT
        h.historical_id,

        COUNT(p.historical_id)
            AS director_prior_films,

        AVG(p.worldwide_box_office)
            AS director_prior_avg_gross,

        MAX(p.worldwide_box_office)
            AS director_prior_max_gross

    FROM historical_movies h

    LEFT JOIN historical_movies p
        ON h.director_key = p.director_key
        AND p.release_year < h.release_year
        AND p.worldwide_box_office > 0

    GROUP BY
        h.historical_id;
    """
)


# TARGET MOVIE DIRECTOR FEATURES

cursor.execute(
    "DROP TABLE IF EXISTS target_director_features;"
)

cursor.execute(
    """
    CREATE TABLE target_director_features AS

    SELECT
        t.target_id,

        COUNT(h.historical_id)
            AS director_prior_films,

        AVG(h.worldwide_box_office)
            AS director_prior_avg_gross,

        MAX(h.worldwide_box_office)
            AS director_prior_max_gross

    FROM prediction_targets t

    LEFT JOIN historical_movies h
        ON t.director_key = h.director_key
        AND h.release_year < t.release_year
        AND h.worldwide_box_office > 0

    GROUP BY
        t.target_id;
    """
)


connection.commit()


director_features = pd.read_sql_query(
    """
    SELECT *
    FROM director_features;
    """,
    connection
)

target_director_features = pd.read_sql_query(
    """
    SELECT *
    FROM target_director_features;
    """,
    connection
)


print("=" * 80)
print("DIRECTOR PERFORMANCE FEATURES CREATED")
print("=" * 80)

print(
    f"Historical feature rows: "
    f"{len(director_features):,}"
)

print(
    f"Target feature rows: "
    f"{len(target_director_features):,}"
)

display(
    target_director_features.head()
)

DIRECTOR PERFORMANCE FEATURES CREATED
Historical feature rows: 7,668
Target feature rows: 19


,target_id,director_prior_films,director_prior_avg_gross,director_prior_max_gross
0,1,2,1.186685e+07,2.208853e+07
1,2,12,1.148616e+08,2.035679e+08
2,3,11,4.685044e+08,1.081143e+09
3,4,3,3.591527e+07,6.015958e+07
4,5,0,NaN,NaN


## 11 LEAD STAR HISTORICAL PERFORMANCE WITH SQLITE

In [12]:
cursor.execute(
    "DROP TABLE IF EXISTS star_features;"
)

cursor.execute(
    """
    CREATE TABLE star_features AS

    SELECT
        h.historical_id,

        COUNT(p.historical_id)
            AS star_prior_films,

        AVG(p.worldwide_box_office)
            AS star_prior_avg_gross,

        MAX(p.worldwide_box_office)
            AS star_prior_max_gross

    FROM historical_movies h

    LEFT JOIN historical_movies p
        ON h.star_key = p.star_key
        AND p.release_year < h.release_year
        AND p.worldwide_box_office > 0

    GROUP BY
        h.historical_id;
    """
)


# TARGET MOVIE STAR FEATURES

cursor.execute(
    "DROP TABLE IF EXISTS target_star_features;"
)

cursor.execute(
    """
    CREATE TABLE target_star_features AS

    SELECT
        t.target_id,

        COUNT(h.historical_id)
            AS star_prior_films,

        AVG(h.worldwide_box_office)
            AS star_prior_avg_gross,

        MAX(h.worldwide_box_office)
            AS star_prior_max_gross

    FROM prediction_targets t

    LEFT JOIN historical_movies h
        ON t.star_key = h.star_key
        AND h.release_year < t.release_year
        AND h.worldwide_box_office > 0

    GROUP BY
        t.target_id;
    """
)


connection.commit()


star_features = pd.read_sql_query(
    """
    SELECT *
    FROM star_features;
    """,
    connection
)

target_star_features = pd.read_sql_query(
    """
    SELECT *
    FROM target_star_features;
    """,
    connection
)


print("=" * 80)
print("LEAD STAR PERFORMANCE FEATURES CREATED")
print("=" * 80)

print(
    f"Historical feature rows: "
    f"{len(star_features):,}"
)

print(
    f"Target feature rows: "
    f"{len(target_star_features):,}"
)

display(
    target_star_features.head()
)

LEAD STAR PERFORMANCE FEATURES CREATED
Historical feature rows: 7,668
Target feature rows: 19


,target_id,star_prior_films,star_prior_avg_gross,star_prior_max_gross
0,1,3,7.180152e+08,1.131928e+09
1,2,0,NaN,NaN
2,3,25,1.616126e+08,6.301624e+08
3,4,8,1.452784e+08,2.780198e+08
4,5,4,3.527871e+07,5.807212e+07


## 12 PRODUCTION COMPANY HISTORICAL PERFORMANCE WITH SQLITE

In [13]:
# HISTORICAL COMPANY FEATURES

cursor.execute(
    "DROP TABLE IF EXISTS company_features;"
)

cursor.execute(
    """
    CREATE TABLE company_features AS

    SELECT
        h.historical_id,

        COUNT(p.historical_id)
            AS company_prior_films,

        AVG(p.worldwide_box_office)
            AS company_prior_avg_gross,

        MAX(p.worldwide_box_office)
            AS company_prior_max_gross

    FROM historical_movies h

    LEFT JOIN historical_movies p
        ON h.company_key = p.company_key
        AND p.release_year < h.release_year
        AND p.worldwide_box_office > 0

    GROUP BY
        h.historical_id;
    """
)


# TARGET COMPANY FEATURES

cursor.execute(
    "DROP TABLE IF EXISTS target_company_features;"
)

cursor.execute(
    """
    CREATE TABLE target_company_features AS

    SELECT
        t.target_id,

        COUNT(h.historical_id)
            AS company_prior_films,

        AVG(h.worldwide_box_office)
            AS company_prior_avg_gross,

        MAX(h.worldwide_box_office)
            AS company_prior_max_gross

    FROM prediction_targets t

    LEFT JOIN historical_movies h
        ON t.company_key = h.company_key
        AND h.release_year < t.release_year
        AND h.worldwide_box_office > 0

    GROUP BY
        t.target_id;
    """
)


connection.commit()


company_features = pd.read_sql_query(
    """
    SELECT *
    FROM company_features;
    """,
    connection
)

target_company_features = pd.read_sql_query(
    """
    SELECT *
    FROM target_company_features;
    """,
    connection
)


print("=" * 80)
print("PRODUCTION COMPANY FEATURES CREATED")
print("=" * 80)

print(
    f"Historical feature rows: "
    f"{len(company_features):,}"
)

print(
    f"Target feature rows: "
    f"{len(target_company_features):,}"
)

display(
    target_company_features.head()
)

PRODUCTION COMPANY FEATURES CREATED
Historical feature rows: 7,668
Target feature rows: 19


,target_id,company_prior_films,company_prior_avg_gross,company_prior_max_gross
0,1,332,1.295450e+08,1.131928e+09
1,2,4,8.660476e+07,2.787804e+08
2,3,1,5.270163e+08,5.270163e+08
3,4,74,5.097942e+07,3.067767e+08
4,5,174,1.142747e+08,1.146031e+09


## 13 MERGE SQL FEATURE BACK INTO MODELLING TABLES

In [14]:

# HISTORICAL FEATURES

historical_features = (
    historical_features
    .merge(
        director_features,
        on="historical_id",
        how="left"
    )
    .merge(
        star_features,
        on="historical_id",
        how="left"
    )
    .merge(
        company_features,
        on="historical_id",
        how="left"
    )
)


# PREDICTION TARGET FEATURES

prediction_features = (
    prediction_features
    .merge(
        target_director_features,
        on="target_id",
        how="left"
    )
    .merge(
        target_star_features,
        on="target_id",
        how="left"
    )
    .merge(
        target_company_features,
        on="target_id",
        how="left"
    )
)


# VALIDATION

print("=" * 80)
print("SQL FEATURES MERGED")
print("=" * 80)

print(
    f"Historical rows: "
    f"{len(historical_features):,}"
)

print(
    f"Prediction rows: "
    f"{len(prediction_features):,}"
)


sql_feature_columns = [
    "director_prior_films",
    "director_prior_avg_gross",
    "director_prior_max_gross",

    "star_prior_films",
    "star_prior_avg_gross",
    "star_prior_max_gross",

    "company_prior_films",
    "company_prior_avg_gross",
    "company_prior_max_gross"
]


print("\nTARGET FEATURE AVAILABILITY")
print("-" * 50)

for column in sql_feature_columns:

    available = (
        prediction_features[column]
        .notna()
        .sum()
    )

    print(
        f"{column:30}: "
        f"{available} / "
        f"{len(prediction_features)}"
    )


display(
    prediction_features[
        [
            "title",
            "director",
            "director_prior_films",
            "director_prior_avg_gross",
            "lead_star",
            "star_prior_films",
            "star_prior_avg_gross",
            "primary_production_company",
            "company_prior_films",
            "company_prior_avg_gross"
        ]
    ]
)

SQL FEATURES MERGED
Historical rows: 7,668
Prediction rows: 19

TARGET FEATURE AVAILABILITY
--------------------------------------------------
director_prior_films          : 19 / 19
director_prior_avg_gross      : 10 / 19
director_prior_max_gross      : 10 / 19
star_prior_films              : 19 / 19
star_prior_avg_gross          : 14 / 19
star_prior_max_gross          : 14 / 19
company_prior_films           : 19 / 19
company_prior_avg_gross       : 16 / 19
company_prior_max_gross       : 16 / 19


,title,director,director_prior_films,director_prior_avg_gross,lead_star,star_prior_films,star_prior_avg_gross,primary_production_company,company_prior_films,company_prior_avg_gross
0,Spider-Man: Brand New Day,Destin Daniel Cretton,2,1.186685e+07,Tom Holland,3,7.180152e+08,Columbia Pictures,332,1.295450e+08
1,Michael,Antoine Fuqua,12,1.148616e+08,Jaafar Jackson,0,NaN,GK Films,4,8.660476e+07
2,The Odyssey,Christopher Nolan,11,4.685044e+08,Matt Damon,25,1.616126e+08,Syncopy,1,5.270163e+08
3,Scary Movie,Michael Tiddes,3,3.591527e+07,Anna Faris,8,1.452784e+08,Miramax,74,5.097942e+07
4,Mortal Kombat II,Simon McQuoid,0,NaN,Karl Urban,4,3.527871e+07,New Line Cinema,174,1.142747e+08
5,The Super Mario Galaxy Movie,Aaron Horvath; Michael Jelenic; Pierre Leduc,0,NaN,Chris Pratt,6,8.797427e+08,Illumination Entertainment,2,1.097122e+09
6,The Devil Wears Prada 2,David Frankel,7,1.156514e+08,Meryl Streep,25,9.141235e+07,Wendy Finerman Productions,0,NaN
7,Toy Story 5,Andrew Stanton,4,6.935936e+08,Tom Hanks,41,2.427420e+08,Pixar Animation Studios,12,6.571954e+08
8,GOAT,Tyree Dillihay,0,NaN,Caleb McLaughlin,0,NaN,Sony Pictures Animation,4,2.319894e+08
9,Backrooms,Kane Parsons,0,NaN,Chiwetel Ejiofor,6,4.158317e+07,A24,8,2.547500e+07


## 14  FINANCIAL TRANSFORMATION

In [15]:
def add_financial_features(df):
    #Add leakage-safe financial transformations.

    #Budget is a predictor.
    #Worldwide box office is the regression target.

    result = df.copy()

    # BUDGET

    result["budget"] = pd.to_numeric(
        result["budget"],
        errors="coerce"
    )

    result["budget_missing"] = (
        result["budget"].isna()
        | result["budget"].le(0)
    ).astype(int)

    result["log_budget"] = np.where(
        result["budget"].gt(0),
        np.log1p(result["budget"]),
        np.nan
    )


    # WORLDWIDE BOX OFFICE

    if "worldwide_box_office" in result.columns:

        result["worldwide_box_office"] = pd.to_numeric(
            result["worldwide_box_office"],
            errors="coerce"
        )

        result["log_worldwide_box_office"] = np.where(
            result["worldwide_box_office"].gt(0),
            np.log1p(
                result["worldwide_box_office"]
            ),
            np.nan
        )

    return result


historical_features = add_financial_features(
    historical_features
)

prediction_features = add_financial_features(
    prediction_features
)


print("=" * 80)
print("FINANCIAL TRANSFORMATIONS")
print("=" * 80)

print(
    "Historical positive budgets:",
    historical_features["budget"].gt(0).sum()
)

print(
    "Historical positive box office:",
    historical_features[
        "worldwide_box_office"
    ].gt(0).sum()
)

print(
    "Prediction target budgets available:",
    prediction_features["budget"].gt(0).sum()
)


display(
    prediction_features[
        [
            "title",
            "budget",
            "budget_missing",
            "log_budget",
            "worldwide_box_office",
            "log_worldwide_box_office"
        ]
    ]
)

FINANCIAL TRANSFORMATIONS
Historical positive budgets: 5497
Historical positive box office: 7479
Prediction target budgets available: 10


,title,budget,budget_missing,log_budget,worldwide_box_office,log_worldwide_box_office
0,Spider-Man: Brand New Day,225000000.0,0,19.231611,2.402937e+09,21.599958
1,Michael,170000000.0,0,18.951309,1.021427e+09,20.744466
2,The Odyssey,250000000.0,0,19.336971,1.627387e+09,21.210241
3,Scary Movie,30000000.0,0,17.216708,2.312778e+08,19.259130
4,Mortal Kombat II,80000000.0,0,18.197537,1.294701e+08,18.678961
5,The Super Mario Galaxy Movie,110000000.0,0,18.515991,1.012793e+09,20.735978
6,The Devil Wears Prada 2,100000000.0,0,18.420681,6.928175e+08,20.356277
7,Toy Story 5,175000000.0,0,18.980297,1.137320e+09,20.851940
8,GOAT,80000000.0,0,18.197537,1.953569e+08,19.090339
9,Backrooms,10000000.0,0,16.118096,3.931673e+08,19.789746


## 15 GENRE STANDARDISATION

In [ ]:
def get_primary_genre(value):"
    #Convert genre values into one comparable primary genre.

    if pd.isna(value):
        return "Unknown"

    genre = str(value).strip()

    if "," in genre:
        genre = genre.split(",")[0]

    genre = genre.strip().title()

    if not genre:
        return "Unknown"

    return genre


historical_features["primary_genre"] = (
    historical_features["genre"]
    .apply(get_primary_genre)
)

prediction_features["primary_genre"] = (
    prediction_features["genre"]
    .apply(get_primary_genre)
)


print("=" * 80)
print("PRIMARY GENRE STANDARDISATION")
print("=" * 80)

print("\nHistorical genres:")
print("-" * 40)

print(
    historical_features[
        "primary_genre"
    ].value_counts()
)


print("\nPrediction target genres:")
print("-" * 40)

print(
    prediction_features[
        "primary_genre"
    ].value_counts()
)


display(
    prediction_features[
        [
            "title",
            "genre",
            "primary_genre"
        ]
    ]
)

## 16 MISSING VALUE STRATEGY

In [ ]:
# NUMERIC FEATURES THAT MAY REQUIRE IMPUTATION LATER

numeric_feature_candidates = [
    "budget",
    "log_budget",

    "release_year",
    "release_month",
    "release_quarter",

    "director_prior_films",
    "director_prior_avg_gross",
    "director_prior_max_gross",

    "star_prior_films",
    "star_prior_avg_gross",
    "star_prior_max_gross",

    "company_prior_films",
    "company_prior_avg_gross",
    "company_prior_max_gross"
]


# ------------------------------------------------------------
# CHECK / ALIGN NUMERIC COLUMNS
#
# If a candidate feature is missing from either dataframe,
# create it as NaN so historical and prediction schemas remain
# compatible.
# ------------------------------------------------------------

print("=" * 80)
print("NUMERIC FEATURE ALIGNMENT")
print("=" * 80)

for column in numeric_feature_candidates:

    historical_exists = (
        column in historical_features.columns
    )

    prediction_exists = (
        column in prediction_features.columns
    )

    print(
        f"{column:30} | "
        f"Historical: {'YES' if historical_exists else 'NO ':3} | "
        f"Prediction: {'YES' if prediction_exists else 'NO ':3}"
    )

    # Create missing columns rather than failing
    if not historical_exists:
        historical_features[column] = np.nan

    if not prediction_exists:
        prediction_features[column] = np.nan


# ADD MISSING-VALUE INDICATORS

for column in numeric_feature_candidates:

    indicator_column = (
        f"{column}_missing"
    )

    historical_features[
        indicator_column
    ] = (
        historical_features[column]
        .isna()
        .astype(int)
    )

    prediction_features[
        indicator_column
    ] = (
        prediction_features[column]
        .isna()
        .astype(int)
    )


# CATEGORICAL MISSING VALUES

categorical_columns = [
    "primary_genre",
    "release_season"
]


for column in categorical_columns:

    # Protect against missing columns
    if column not in historical_features.columns:
        historical_features[column] = "Unknown"

    if column not in prediction_features.columns:
        prediction_features[column] = "Unknown"

    historical_features[column] = (
        historical_features[column]
        .fillna("Unknown")
        .astype(str)
    )

    prediction_features[column] = (
        prediction_features[column]
        .fillna("Unknown")
        .astype(str)
    )


# VALIDATION

print("\n" + "=" * 80)
print("MISSING-VALUE STRATEGY")
print("=" * 80)

print(
    """
Numeric values:
- Missing values retained as NaN
- Missing-value indicator columns created
- No global numerical imputation performed
- Numerical imputation deferred to the modelling pipeline

Categorical values:
- Missing values replaced with 'Unknown'

Reason:
Imputation statistics must be learned from training data only
to avoid data leakage.
"""
)


print("=" * 80)
print("MISSING VALUES BY FEATURE")
print("=" * 80)

missing_summary = []

for column in numeric_feature_candidates:

    missing_summary.append({
        "feature": column,

        "historical_missing":
            historical_features[
                column
            ].isna().sum(),

        "prediction_missing":
            prediction_features[
                column
            ].isna().sum()
    })


missing_summary = pd.DataFrame(
    missing_summary
)

display(missing_summary)

NUMERIC FEATURE ALIGNMENT
budget                         | Historical: YES | Prediction: YES
log_budget                     | Historical: YES | Prediction: YES
release_year                   | Historical: YES | Prediction: YES
release_month                  | Historical: YES | Prediction: YES
release_quarter                | Historical: YES | Prediction: YES
director_prior_films           | Historical: YES | Prediction: YES
director_prior_avg_gross       | Historical: YES | Prediction: YES
director_prior_max_gross       | Historical: YES | Prediction: YES
star_prior_films               | Historical: YES | Prediction: YES
star_prior_avg_gross           | Historical: YES | Prediction: YES
star_prior_max_gross           | Historical: YES | Prediction: YES
company_prior_films            | Historical: YES | Prediction: YES
company_prior_avg_gross        | Historical: YES | Prediction: YES
company_prior_max_gross        | Historical: YES | Prediction: YES

MISSING-VALUE STRATEGY

Numeric val

,feature,historical_missing,prediction_missing
0,budget,2171,9
1,log_budget,2171,9
2,release_year,0,0
3,release_month,59,0
4,release_quarter,59,0
5,director_prior_films,0,0
6,director_prior_avg_gross,3021,9
7,director_prior_max_gross,3021,9
8,star_prior_films,0,0
9,star_prior_avg_gross,2959,5


## 17 FINAL FEATURE SELECTION

In [19]:
primary_numeric_features = [

    "log_budget",

    "release_year",
    "release_month",
    "release_quarter",
    "holiday_release",

    "director_prior_films",
    "director_prior_avg_gross",

    "star_prior_films",
    "star_prior_avg_gross",

    "company_prior_films",
    "company_prior_avg_gross"
]


primary_categorical_features = [

    "primary_genre",
    "release_season"
]


primary_model_features = (
    primary_numeric_features
    + primary_categorical_features
)


regression_target = (
    "log_worldwide_box_office"
)


identifier_columns = [
    "title",
    "release_date"
]


print("=" * 80)
print("PRIMARY MODEL FEATURE SET")
print("=" * 80)

print("\nNUMERIC FEATURES")
print("-" * 40)

for feature in primary_numeric_features:
    print(feature)


print("\nCATEGORICAL FEATURES")
print("-" * 40)

for feature in primary_categorical_features:
    print(feature)


print("\nREGRESSION TARGET")
print("-" * 40)

print(regression_target)

PRIMARY MODEL FEATURE SET

NUMERIC FEATURES
----------------------------------------
log_budget
release_year
release_month
release_quarter
holiday_release
director_prior_films
director_prior_avg_gross
star_prior_films
star_prior_avg_gross
company_prior_films
company_prior_avg_gross

CATEGORICAL FEATURES
----------------------------------------
primary_genre
release_season

REGRESSION TARGET
----------------------------------------
log_worldwide_box_office


## 18 ALIGN HISTORICAL AND PREDICTION SCHEMAS

In [20]:

# HISTORICAL MODELLING TABLE

historical_model_data = (
    historical_features[
        identifier_columns
        + primary_model_features
        + [
            "worldwide_box_office",
            "log_worldwide_box_office"
        ]
    ]
    .copy()
)


# Only rows with a valid regression target
historical_model_data = (
    historical_model_data[
        historical_model_data[
            "log_worldwide_box_office"
        ].notna()
    ]
    .reset_index(drop=True)
)


# PREDICTION MODELLING TABLE

prediction_model_data = (
    prediction_features[
        [
            "title",
            "cohort",
            "release_date"
        ]
        + primary_model_features
        + [
            "worldwide_box_office",
            "log_worldwide_box_office"
        ]
    ]
    .copy()
)


# VERIFY FEATURE ALIGNMENT

historical_feature_set = set(
    primary_model_features
)

prediction_feature_set = set(
    primary_model_features
)

schema_match = (
    historical_feature_set
    == prediction_feature_set
)


print("=" * 80)
print("MODEL SCHEMA ALIGNMENT")
print("=" * 80)

print(
    f"Historical modelling rows: "
    f"{len(historical_model_data):,}"
)

print(
    f"Prediction target rows: "
    f"{len(prediction_model_data):,}"
)

print(
    f"Model features: "
    f"{len(primary_model_features)}"
)

print(
    f"Feature schemas aligned: "
    f"{schema_match}"
)


display(
    historical_model_data.head()
)

display(
    prediction_model_data.head()
)

MODEL SCHEMA ALIGNMENT
Historical modelling rows: 7,479
Prediction target rows: 19
Model features: 13
Feature schemas aligned: True


,title,release_date,log_budget,release_year,release_month,release_quarter,holiday_release,director_prior_films,director_prior_avg_gross,star_prior_films,star_prior_avg_gross,company_prior_films,company_prior_avg_gross,primary_genre,release_season,worldwide_box_office,log_worldwide_box_office
0,The Shining,1980-06-13,16.759950,1980,6.0,2.0,0,0,NaN,0,NaN,0,NaN,Unknown,summer,46998772.0,17.665632
1,The Blue Lagoon,1980-07-02,15.319588,1980,7.0,3.0,0,0,NaN,0,NaN,0,NaN,Unknown,summer,58853106.0,17.890555
2,Star Wars: Episode V - The Empire Strikes Back,1980-06-20,16.705882,1980,6.0,2.0,0,0,NaN,0,NaN,0,NaN,Unknown,summer,538375067.0,20.104066
3,Airplane!,1980-07-02,15.068274,1980,7.0,3.0,0,0,NaN,0,NaN,0,NaN,Unknown,summer,83453539.0,18.239801
4,Caddyshack,1980-07-25,15.607270,1980,7.0,3.0,0,0,NaN,0,NaN,0,NaN,Unknown,summer,39846344.0,17.500541


,title,cohort,release_date,log_budget,release_year,release_month,release_quarter,holiday_release,director_prior_films,director_prior_avg_gross,star_prior_films,star_prior_avg_gross,company_prior_films,company_prior_avg_gross,primary_genre,release_season,worldwide_box_office,log_worldwide_box_office
0,Spider-Man: Brand New Day,released_backtest,2026-07-31,19.231611,2026,7,3,0,2,1.186685e+07,3,7.180152e+08,332,1.295450e+08,Unknown,summer,2.402937e+09,21.599958
1,Michael,released_backtest,2026-04-24,18.951309,2026,4,2,0,12,1.148616e+08,0,NaN,4,8.660476e+07,Unknown,spring,1.021427e+09,20.744466
2,The Odyssey,released_backtest,2026-07-17,19.336971,2026,7,3,0,11,4.685044e+08,25,1.616126e+08,1,5.270163e+08,Unknown,summer,1.627387e+09,21.210241
3,Scary Movie,released_backtest,2026-06-05,17.216708,2026,6,2,0,3,3.591527e+07,8,1.452784e+08,74,5.097942e+07,Unknown,summer,2.312778e+08,19.259130
4,Mortal Kombat II,released_backtest,2026-05-08,18.197537,2026,5,2,0,0,NaN,4,3.527871e+07,174,1.142747e+08,Unknown,summer,1.294701e+08,18.678961


## 19 FINAL VALIDATION AND SAVE FEATURE TABLES

In [21]:
# VALIDATION

print("=" * 80)
print("FINAL FEATURE ENGINEERING VALIDATION")
print("=" * 80)


print(
    f"Historical rows: "
    f"{len(historical_model_data):,}"
)

print(
    f"Prediction rows: "
    f"{len(prediction_model_data):,}"
)

print(
    f"Historical duplicate titles: "
    f"{historical_model_data['title'].duplicated().sum():,}"
)

print(
    f"Prediction duplicate titles: "
    f"{prediction_model_data['title'].duplicated().sum():,}"
)


future_outcome_count = (
    prediction_model_data.loc[
        prediction_model_data[
            "cohort"
        ].eq(
            "future_prediction"
        ),
        "worldwide_box_office"
    ]
    .notna()
    .sum()
)


print(
    f"Future movies with known outcome: "
    f"{future_outcome_count}"
)


# SAVE CSV FILES

historical_model_data.to_csv(
    HISTORICAL_FEATURE_PATH,
    index=False
)

prediction_model_data.to_csv(
    TARGET_FEATURE_PATH,
    index=False
)


# SAVE FINAL SQLITE FEATURE TABLES

historical_model_data.to_sql(
    "historical_model_features",
    connection,
    if_exists="replace",
    index=False
)

prediction_model_data.to_sql(
    "prediction_model_features",
    connection,
    if_exists="replace",
    index=False
)


connection.commit()


print("\n" + "=" * 80)
print("FEATURE OUTPUTS SAVED")
print("=" * 80)

print("\nHistorical features:")
print(HISTORICAL_FEATURE_PATH)

print("\nPrediction features:")
print(TARGET_FEATURE_PATH)

print("\nSQLite database:")
print(FEATURE_DB_PATH)

FINAL FEATURE ENGINEERING VALIDATION
Historical rows: 7,479
Prediction rows: 19
Historical duplicate titles: 147
Prediction duplicate titles: 0
Future movies with known outcome: 0

FEATURE OUTPUTS SAVED

Historical features:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Features\historical_model_features.csv

Prediction features:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Features\prediction_target_features.csv

SQLite database:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Database\movie_features.db
